In [10]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin


# 爬取百度热搜


In [11]:

#1.百度热搜页面地址
url = "https://top.baidu.com/board?tab=realtime"

#2.添加请求头
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}

#3.发送网页请求
response = requests.get(url, headers=headers, timeout=10)
#4.设置中文编码
response.encoding="utf-8"

# 5. 检查网页是否访问成功
print("网页状态码:", response.status_code)


网页状态码: 200


In [12]:
#6.获取HTML源码
html = response.text
print("HTML源码前1000字符:", html[:1000])

#7.用BeautifulSoup解析HTML
soup = BeautifulSoup(html, "lxml")
print("解析后的HTML前1000字符:", soup.prettify()[:1000])

#8.找到每一条热搜的外层容器
items = soup.select("div[class*='category-wrap']")
print("热搜条数:", len(items))



HTML源码前1000字符: 
        <!DOCTYPE html>
        <html>
            <head>
                <meta http-equiv="Content-Type" content="text/html;charset=utf-8">
                <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
                <meta content="always" name="referrer">
                <meta name="theme-color" content="#2932e1">
                <link rel="shortcut icon" href="//www.baidu.com/favicon.ico" type="image/x-icon" />
                <link rel="icon" sizes="any" mask href="//www.baidu.com/img/baidu_85beaf5496f291521eb75ba38eacbd87.svg">
                <link rel="dns-prefetch" href="//fyb-pc-static.cdn.bcebos.com"/>
                <meta name="keywords" content="百度热搜,百度热搜榜,百度搜索排行榜,搜索排行榜,百度热门搜索,今日热搜,今日热点,排行榜,热搜榜,热词榜,热门话题,网络热点,实时热点,热门事件,热点">
                <meta name="description" content="百度热搜以数亿用户海量的真实数据为基础，通过专业的数据挖掘方法，计算关键词的热搜指数，旨在建立权威、全面、热门、时效的各类关键词排行榜，引领热词阅读时代。">
                <title>百度热搜</title>
                <style data-vue-ssr-id="22cfed39:0">
.

In [14]:

#9.保存解析结果
hot_list = []

for item in items[:10]:
   #热搜名称
   title_tag = item.select_one("div[class*='c-single-text-ellipsis']")

   #热度值
   hot_tag = item.select_one("div[class*='hot-index']")

   #链接
   link_tag = item.select_one("a[href]")

   keyword = title_tag.get_text(strip=True) if title_tag else ""  
   hot_score = hot_tag.get_text(strip=True) if hot_tag else ""  
   link = link_tag.get("href") if link_tag else ""  

   #把相对链接转成完整链接
   if link:  
      link = urljoin(url, link) 
   
   #爬取日期
   crawl_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

   #.保存到列表
   hot_list.append({  
       "keyword": keyword,  
       "hot_score": hot_score,  
       "url": link,  
       "crawl_date": crawl_date
})  

print("百度热搜 Top10：")

for item in hot_list:
   print("热搜名称:", item["keyword"])
   print("热度值:", item["hot_score"])
   print("链接:", item["url"])
   print("日期:", item["crawl_date"])
   print("-" * 50)

print("共爬取:", len(hot_list), "条")

百度热搜 Top10：
热搜名称: 水润家园好光景
热度值: 7904504
链接: https://www.baidu.com/s?wd=%E6%B0%B4%E6%B6%A6%E5%AE%B6%E5%9B%AD%E5%A5%BD%E5%85%89%E6%99%AF&sa=fyb_news&rsv_dl=fyb_news
日期: 2026-06-04 16:58:24
--------------------------------------------------
热搜名称: 辽宁省长：严肃查处相关单位和责任人
热度值: 7808639
链接: https://www.baidu.com/s?wd=%E8%BE%BD%E5%AE%81%E7%9C%81%E9%95%BF%EF%BC%9A%E4%B8%A5%E8%82%83%E6%9F%A5%E5%A4%84%E7%9B%B8%E5%85%B3%E5%8D%95%E4%BD%8D%E5%92%8C%E8%B4%A3%E4%BB%BB%E4%BA%BA&sa=fyb_news&rsv_dl=fyb_news
日期: 2026-06-04 16:58:24
--------------------------------------------------
热搜名称: 从怀孕到生娃能领7笔钱
热度值: 7714619
链接: https://www.baidu.com/s?wd=%E4%BB%8E%E6%80%80%E5%AD%95%E5%88%B0%E7%94%9F%E5%A8%83%E8%83%BD%E9%A2%867%E7%AC%94%E9%92%B1&sa=fyb_news&rsv_dl=fyb_news
日期: 2026-06-04 16:58:24
--------------------------------------------------
热搜名称: 端午火车票明日开售
热度值: 7618746
链接: https://www.baidu.com/s?wd=%E7%AB%AF%E5%8D%88%E7%81%AB%E8%BD%A6%E7%A5%A8%E6%98%8E%E6%97%A5%E5%BC%80%E5%94%AE&sa=fyb_news&rsv_dl=fyb_news
日期: 2026-06

# 连接MySQL数据库

In [15]:
import pymysql


In [16]:
# 连接mysql
conn = pymysql.connect(
    host="localhost",
    port=3306,
    user="root",
    password="shq20030214", #你的MySQL密码
    database="web_crawler_db",
    charset="utf8mb4"
)

In [17]:
# 创建 cursor，cursor 是执行 SQL 的对象
cursor = conn.cursor()

# 10. 准备 INSERT SQL
sql = """
INSERT INTO baidu_hot_search
(keyword, hot_score, url, crawl_date)
VALUES (%s, %s, %s, %s)
"""

In [18]:
# 11. 把字典列表转换成 tuple 列表
data = []

for item in hot_list:
    data.append((
        item["keyword"],
        item["hot_score"],
        item["url"],
        item["crawl_date"]
    ))

# 12. 批量插入 MySQL
cursor.executemany(sql, data)

# 13. 提交事务，真正保存到数据库
conn.commit()

# 14. 关闭 cursor 和连接
cursor.close()
conn.close()

print("百度热搜数据已成功写入 MySQL")

百度热搜数据已成功写入 MySQL


# 通过mysqlhelper连接MYsql数据库

In [19]:
from baidu_hot_mysqlhelper import MySqlHelper

In [20]:
db = MySqlHelper(
    host="localhost",
    port=3306,
    user="root",
    password="shq20030214", #你的MySQL密码
    database="web_crawler_db"
)

# 6. 准备 SQL
sql = """
INSERT INTO baidu_hot_search
(keyword, hot_score, url, crawl_date)
VALUES (%s, %s, %s, %s)
"""

# 7. 准备插入数据
data = []

for item in hot_list:
    data.append((
        item["keyword"],
        item["hot_score"],
        item["url"],
        item["crawl_date"]
    ))

# 8. 调用 MySqlHelper 批量插入
db.executemany(sql, data)

print("百度热搜数据已通过 MySqlHelper 成功写入 MySQL")

百度热搜数据已通过 MySqlHelper 成功写入 MySQL
